In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
import numpy as np

class EnhancedCropPreprocessor:
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.le = LabelEncoder()
        
        self.numeric_features = [
            'soil_ph', 'nitrogen_kg_ha', 'phosphorus_kg_ha', 'potassium_kg_ha',
            'annual_rainfall_mm', 'avg_temp_c', 'avg_humidity_pct'
        ]
        self.categorical_features = ['soil_type', 'irrigation_type', 'previous_crop']
        self.features = self.numeric_features + self.categorical_features
        self.target = 'recommended_crop'

    def preprocess(self, df: pd.DataFrame):
        print("Starting the work: Cleaning and Balancing Data...")

        # 1. Feature Selection
        df_model = df[self.features + [self.target]].copy()

        # --------------------------------------------------------
        # --- START OF NEW FEATURE ENGINEERING  ---
        # --------------------------------------------------------
        
        df_model['phosphorus_kg_ha'] = df_model['phosphorus_kg_ha'].replace(0, 1)
        df_model['potassium_kg_ha'] = df_model['potassium_kg_ha'].replace(0, 1)
        df_model['soil_ph'] = df_model['soil_ph'].replace(0, 0.01) 
        df_model['avg_humidity_pct'] = df_model['avg_humidity_pct'].replace(0, 1)

        df_model['N_ratio_P'] = df_model['nitrogen_kg_ha'] / df_model['phosphorus_kg_ha']
        df_model['P_ratio_K'] = df_model['phosphorus_kg_ha'] / df_model['potassium_kg_ha']

        df_model['moisture_stress'] = df_model['avg_temp_c'] / df_model['avg_humidity_pct']

        df_model['acidity_index'] = df_model['annual_rainfall_mm'] / df_model['soil_ph']

        
        self.numeric_features.extend(['N_ratio_P', 'P_ratio_K', 'moisture_stress', 'acidity_index'])
        # ------------------------------------------------------
        # --- END OF NEW FEATURE ENGINEERING ---
        # ------------------------------------------------------

        df_encoded = pd.get_dummies(df_model, columns=self.categorical_features, drop_first=True)

        X = df_encoded.drop(columns=[self.target])
        Y = df_encoded[self.target]
        
        Y_encoded = self.le.fit_transform(Y)
        print(f"Total crop types (labels): {len(self.le.classes_)}")

        X_train, X_test, Y_train, Y_test = train_test_split(
            X, Y_encoded, test_size=0.2, random_state=self.random_state, stratify=Y_encoded
        )
        
        # Scaling 
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()

        X_train_scaled[self.numeric_features] = self.scaler.fit_transform(X_train[self.numeric_features])
        X_test_scaled[self.numeric_features] = self.scaler.transform(X_test[self.numeric_features])

        # Apply SMOTE 
        smote = SMOTE(sampling_strategy='auto', random_state=self.random_state)
        X_train_smote, Y_train_smote = smote.fit_resample(X_train_scaled, Y_train)
        
        print(f"Practice data size before SMOTE: {len(X_train_scaled)}")
        print(f"Practice data size AFTER SMOTE: {len(X_train_smote)} (Now balanced)")
        
        # Return the new, enhanced, and scaled datasets
        return X_train_smote, X_test_scaled, Y_train_smote, Y_test, self.le, self.scaler, X.columns.tolist()

In [ ]:
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from scipy.stats import randint, uniform

class ImprovedModelEvaluator:
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.le = None
        self.scaler = None
        self.all_model_columns = None
        self.best_model = None

    def train_stacking_model(self, X_train, X_test, Y_train, Y_test):
        
        print("\nStarting work: Building the Stacking Model...")

        # 1. Define the "Base Trainers" 
        base_estimators = [
            # base 1: Random Forest 
            ('rf', RandomForestClassifier(random_state=self.random_state)),
            # base 2: Gradient Boosting
            ('gb', GradientBoostingClassifier(random_state=self.random_state))
        ]

        # 2. Define the "Robot Boss" (The Meta-Classifier that makes the final decision)
        # We'll use a simple, fast classifier for the boss.
        final_estimator = RandomForestClassifier(random_state=self.random_state, n_estimators=50)

        # 3. Create the Stacking Model
        stacking_clf = StackingClassifier(
            estimators=base_estimators, 
            final_estimator=final_estimator,
            cv=StratifiedKFold(n_splits=3), # Use 3-fold cross-validation for speed during stacking
            n_jobs=-1 # Use all computer cores
        )

        # 4. Define settings (Hyperparameters) for Randomized Search
        param_grid_stack = {
            # Settings for the Random Forest trainer
            'rf__n_estimators': randint(50, 200),
            'rf__max_depth': randint(5, 15),
            # Settings for the Gradient Boosting trainer
            'gb__n_estimators': randint(50, 200),
            'gb__learning_rate': uniform(0.01, 0.2),
        }
        
        # 5. Run the Randomized Search (The quick way to find the best settings!)
        # We optimize for 'f1_weighted' because it makes sure the rare crops get predicted well.
        search = RandomizedSearchCV(
            estimator=stacking_clf,
            param_distributions=param_grid_stack,
            n_iter=20, 
            scoring='f1_weighted', 
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=self.random_state),
            verbose=1,
            n_jobs=-1,
            random_state=self.random_state
        )
        
        # Train the model!
        search.fit(X_train, Y_train)
        
        self.best_model = search.best_estimator_
        
        # 6. Final Evaluation
        y_pred = self.best_model.predict(X_test)
        
        print("\n--- Final Model Results ---")
        print(f"Best cross-validation score (F1-Weighted): {search.best_score_:.4f}")
        print(f"Test set Accuracy: {accuracy_score(Y_test, y_pred):.4f}")
        print("\nClassification Report (The report card for each crop):")
        print(classification_report(Y_test, y_pred, target_names=self.le.classes_))
        
        print("\nTraining complete!")
        return self.best_model
    
    def save_model(self, model, filename='best_stacking_model.joblib'):
        """Saves the final model."""
        joblib.dump(model, filename)
        joblib.dump(self.scaler, 'scaler.joblib')
        joblib.dump(self.le, 'encoder.joblib')
        print(f"Saved the final model to '{filename}', scaler, and encoder.")

    def recommend_crop_final(self, user_input: dict):
        """The function to use the final model to give a recommendation."""
        
        # 1. Create a dummy dataframe with all 25 columns the model expects
        input_df = pd.DataFrame(columns=self.all_model_columns, index=[0]).fillna(0)
        
        # 2. Fill in the user's input values
        for col in self.numeric_features:
            input_df.loc[0, col] = user_input.get(col, 0)
            
        # 3. Handle the categorical inputs (soil type, etc.)
        for cat_feature in self.categorical_features:
            value = user_input.get(cat_feature)
            if value:
                oh_column = f'{cat_feature}_{value}'
                if oh_column in input_df.columns:
                    input_df.loc[0, oh_column] = 1
        
        # Ensure correct data types and scale the numeric features
        input_df = input_df.astype(float)
        input_df[self.numeric_features] = self.scaler.transform(input_df[self.numeric_features])
        
        # 4. Predict and decode the result
        prediction_label = self.best_model.predict(input_df)[0]
        recommended_crop = self.le.inverse_transform([prediction_label])[0]
        
        return recommended_crop

In [ ]:
# --- Main Execution Block (MODIFIED FOR JUPYTER) ---

# We define these variables globally to be used in the saving cell later
global_evaluator = None 
global_best_model = None

def main():
    """Tells the and trainer to start the work!"""
    data_path = 'crop_data.csv'

    # Initialize the main classes
    evaluator = ImprovedModelEvaluator(random_state=42)
    preprocessor = EnhancedCropPreprocessor(random_state=42)

    try:
        # 1. Load and Preprocess Data (The  's Work)
        df = pd.read_csv(data_path)
        X_train, X_test, Y_train, Y_test, le, scaler, all_model_columns = preprocessor.preprocess(df)

        # Pass the necessary objects to the evaluator
        evaluator.le = le
        evaluator.scaler = scaler
        evaluator.all_model_columns = all_model_columns

        # 2. Train the Model (The Trainer's Work)
        best_model = evaluator.train_stacking_model(X_train, X_test, Y_train, Y_test)

        # 3. CRITICAL CHANGE: EXPOSE OBJECTS GLOBALLY
        # This makes the variables accessible to the next cell!
        global global_evaluator, global_best_model
        global_evaluator = evaluator
        global_best_model = best_model

        print("\nModel trained successfully. Proceed to the next cell to save.")

    except FileNotFoundError:
        print(f"Oops! File '{data_path}' not found.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == '__main__':
    main()

# IMPORTANT: Run this entire block first.

In [ ]:
import joblib
import os

# --- 1. Define Model and Accessory Names ---
model_filename = 'best_stacking_model_final.joblib'
scaler_filename = 'scaler_final.joblib'
encoder_filename = 'encoder_final.joblib'

# --- 2. Save Files ---
try:
    if global_best_model is not None:
        # This path will point to the directory where your Jupyter notebook file is located.
        save_path = os.getcwd() 
        print(f"Saving files to the current directory: {save_path}")

        # Save the model
        joblib.dump(global_best_model, os.path.join(save_path, model_filename))
        
        # Save the scaler and encoder using the exposed evaluator object
        joblib.dump(global_evaluator.scaler, os.path.join(save_path, scaler_filename))
        joblib.dump(global_evaluator.le, os.path.join(save_path, encoder_filename))
        
        print("\n Success! All model files have been saved.")
        print(f"Files saved: {model_filename}, {scaler_filename}, {encoder_filename}")
        print("You can find them in the same folder as this Jupyter Notebook.")
    else:
        print("ERROR: The model was not successfully trained (global_best_model is None).")

except NameError:
    print("FATAL ERROR: The global variables were not created. Did you run the modified training cell?")
except Exception as e:
    print(f"An unexpected error occurred during file saving: {e}")

In [ ]:
# import pandas as pd
# import joblib
# import os
# import numpy as np
# from sklearn.base import BaseEstimator, ClassifierMixin

# # --- Define Classes Needed for Joblib to Load ---
# # IMPORTANT: These classes must be defined so joblib can load the model components.

# # Re-define the base classes used in your StackingClassifier and Preprocessor
# class EnhancedCropPreprocessor:
#     # Minimal definition required for joblib to load the encoder/scaler
#     def __init__(self):
#         self.numeric_features = [
#             'soil_ph', 'nitrogen_kg_ha', 'phosphorus_kg_ha', 'potassium_kg_ha',
#             'annual_rainfall_mm', 'avg_temp_c', 'avg_humidity_pct'
#         ]
#         self.categorical_features = ['soil_type', 'irrigation_type', 'previous_crop']
#         self.features = self.numeric_features + self.categorical_features
#         self.target = 'recommended_crop'


# class SimplePredictor:
#     # Dummy class to hold the prediction logic
#     def __init__(self, model, scaler, le, all_model_columns):
#         self.model = model
#         self.scaler = scaler
#         self.le = le
#         self.all_model_columns = all_model_columns
#         self.numeric_features = ['soil_ph', 'nitrogen_kg_ha', 'phosphorus_kg_ha', 'potassium_kg_ha',
#                                  'annual_rainfall_mm', 'avg_temp_c', 'avg_humidity_pct']
#         self.categorical_features = ['soil_type', 'irrigation_type', 'previous_crop']


#     def recommend_crop_final(self, user_input: dict):
#         """
#         Loads the trained model and predicts the recommended crop based on user input.
#         """
        
#         # 1. Create a dummy dataframe with all 25 columns the model expects (filled with zeros)
#         input_df = pd.DataFrame(columns=self.all_model_columns, index=[0]).fillna(0)
        
#         # 2. Populate input_df with user's numerical features
#         for col in self.numeric_features:
#             input_df.loc[0, col] = user_input.get(col, 0)
            
#         # 3. Handle one-hot encoded categorical features
#         for cat_feature in self.categorical_features:
#             value = user_input.get(cat_feature)
#             if value:
#                 # Construct the one-hot column name (e.g., 'soil_type_Sandy')
#                 oh_column = f'{cat_feature}_{value}'
#                 if oh_column in input_df.columns:
#                     input_df.loc[0, oh_column] = 1
        
#         # Ensure all data types are float
#         input_df = input_df.astype(float)
        
#         # 4. Scaling: Apply the SAVED scaler to the new input data
#         input_df[self.numeric_features] = self.scaler.transform(input_df[self.numeric_features])
        
#         # 5. Prediction
#         prediction_label = self.model.predict(input_df)[0]
        
#         # 6. Convert numerical label back to crop name
#         recommended_crop = self.le.inverse_transform([prediction_label])[0]
        
#         return recommended_crop


# # --- MAIN EXECUTION BLOCK (FOR DEPLOYMENT) ---
# if __name__ == '__main__':
    
#     # Define file paths based on the successful save names
#     MODEL_PATH = 'best_stacking_model_final.joblib'
#     SCALER_PATH = 'scaler_final.joblib'
#     ENCODER_PATH = 'encoder_final.joblib'
    
#     try:
#         # 1. Load the trained components
#         model = joblib.load(MODEL_PATH)
#         scaler = joblib.load(SCALER_PATH)
#         le = joblib.load(ENCODER_PATH)
        
#         # NOTE: We need the list of column names the model was trained on. 
#         # Since we cannot load this from the joblib file, we infer it from the scaler.
#         # This is a common deployment hack.
#         # For a truly robust deployment, save the all_model_columns list to a separate file.
        
#         # For simplicity, we'll recreate a generic list of columns using common features:
#         # In a real setup, load 'all_model_columns' from a saved text file.
        
#         print("Model components loaded successfully. Preparing prediction script...")

#         # --- IMPORTANT: Get all 25 column names from a dummy run or saved list ---
#         # Since the column list was not saved, we will load your crop_data.csv 
#         # one last time to get the correct column structure.
#         df = pd.read_csv('crop_data.csv')
#         temp_preprocessor = EnhancedCropPreprocessor()
#         df_model = df[temp_preprocessor.features + [temp_preprocessor.target]].copy()
#         df_encoded = pd.get_dummies(df_model, columns=temp_preprocessor.categorical_features, drop_first=True)
#         all_model_columns = df_encoded.drop(columns=[temp_preprocessor.target]).columns.tolist()


#         # 2. Initialize the Predictor Class
#         predictor = SimplePredictor(model, scaler, le, all_model_columns)
        
#         # --- 3. Run a New Prediction Test ---
        
#         # Test Case: High-rain, low-nutrient conditions (often suitable for rice or maize)
#         test_input = {
#             'soil_ph': 5.5,
#             'nitrogen_kg_ha': 80,
#             'phosphorus_kg_ha': 40,
#             'potassium_kg_ha': 35,
#             'annual_rainfall_mm': 1800,
#             'avg_temp_c': 27.5,
#             'avg_humidity_pct': 85,
#             'soil_type': 'Loamy',
#             'irrigation_type': 'Rainfed',
#             'previous_crop': 'Dal'
#         }
        
#         recommendation = predictor.recommend_crop_final(test_input)
        
#         print("\n--- Final Recommendation from Saved Model ---")
#         print(f"Input Conditions: High Rain, Low pH (Acidic Soil), Rainfed Irrigation.")
#         print(f"Recommended Crop: \033[1m{recommendation}\033[0m")
        
#     except FileNotFoundError:
#         print(f"\nFATAL ERROR: Could not find one or more files.")
#         print("Please ensure the following files are in this directory:")
#         print(f"- {MODEL_PATH}\n- {SCALER_PATH}\n- {ENCODER_PATH}")
#     except Exception as e:
#         print(f"\nAn unexpected error occurred during prediction: {e}")